# 05 – Directly mapped types and final output

Reads the output of **04** (`nisjemodel-remote-sensing_norge_2026_25833.geo.parquet`)
and overrides the classification where directly mapped areas exist:

1. **NA-MC08 – Marint bunnsystem preget av oksygenmangel**
   from the MDir water type classification `NyTypologi2022`, types **1a, 2a, 3a**
   (`… med oksygenfattig bunnvann`, see `mnk.rules.WATER_TYPE_NAMES`).
   Only bottom with `nkm_lys` afotisk or eufotisk and `nkm_lm-tv = LM-TV_0` (permanently submerged, not intertidal).
2. **NA-MJ01 – Taretrålingsbunn**
   from NIVA kelp trawling areas 2020–2025. Applies to all bottom inside the areas.

The rules are defined in `mnk.rules.DIRECT_RULES`. Layers are applied in order; a later layer overrides an earlier one where they overlap.
Only affected polygons (spatial index) go through geometry operations.

The ungridded result is saved as `marint-naturkart-ungridded_norge_2026_25833.geo.parquet`
(source for PMTiles). Finally everything is split along the SSB 1 km grid (`rute_fkid`) and written as
`marint-naturkart_norge_2026_25833` (parquet + gpkg).

In [1]:
import gc
import json
import tempfile
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

IN_PARQUET = REPO / f"{mnk.utils.to_filename('nisjemodel-remote-sensing', 'norge', '2026', '25833')}.geo.parquet"
GRID_URL = "gs://niva-geodata/MarintNaturKart/aux/Basisdata_0000_Norge_25833_StatistiskRutenett1km.geo.parquet"

for rule in mnk.rules.DIRECT_RULES:
    print(rule["nin_kode_fkid"], mnk.rules.NIN_NAMES[rule["nin_kode_fkid"]], rule["url"])

NA-MC08 Marint bunnsystem preget av oksygenmangel gs://niva-geodata/MarintNaturKart/input/mdir/NyTypologi2022.geo.parquet
NA-MJ01 Taretrålingsbunn gs://niva-geodata/MarintNaturKart/input/niva/taretraling_polygons_2020-2025_2026-11-15.gpkg


## 1. Load data

In [2]:
final = gpd.read_parquet(IN_PARQUET)
assert final.crs.to_epsg() == 25833
print(f"Input (04): {len(final):,} polygons")


def load_mask(rule):
    """Read the direct dataset of *rule* and return clean single polygons."""
    url = rule["url"]
    gdf = gpd.read_parquet(url) if url.endswith(".parquet") else gpd.read_file(url)
    if rule["column"] is not None:
        gdf = gdf.loc[gdf[rule["column"]].isin(rule["values"])]
    gdf = gdf[["geometry"]].to_crs(CRS)
    gdf = gdf.set_geometry(shapely.make_valid(gdf.geometry.to_numpy()), crs=CRS)
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[gdf.geom_type == "Polygon"].reset_index(drop=True)
    print(f"{rule['nin_kode_fkid']}: {len(gdf):,} polygons ({gdf.area.sum() / 1e6:,.1f} km²)")
    return gdf

Input (04): 4,977,343 polygons


## 2. Generic override

`override` splits affected polygons into the part inside the mask (new type) and
the remainder (unchanged), same pattern as in 04.

In [3]:
def override(final, mask, code, method, candidates=None):
    """Assign NiN type *code* to the parts of *final* inside *mask*."""
    if candidates is None:
        candidates = pd.Series(True, index=final.index)
    sub = final.loc[candidates]
    hit = sub.sindex.query(mask.geometry, predicate="intersects")
    hit_idx = sub.index[np.unique(hit[1])]

    affected = final.loc[hit_idx].copy()
    unaffected = final.drop(index=hit_idx)
    affected["_src"] = np.arange(len(affected))

    pairs = gpd.sjoin(affected, mask[["geometry"]], how="inner", predicate="intersects")
    mask_geom = mask.geometry.to_numpy()[pairs["index_right"].to_numpy()]
    inter = shapely.intersection(pairs.geometry.to_numpy(), mask_geom)
    inside = pairs.set_geometry(inter, crs=CRS).drop(columns="index_right")
    inside = inside.explode(index_parts=False)
    inside = inside[(inside.geom_type == "Polygon") & ~inside.geometry.is_empty]
    # Overlapping mask polygons give duplicates – dissolve per source polygon.
    inside = inside.dissolve(by="_src", as_index=False).explode(index_parts=False)
    inside = inside[inside.geom_type == "Polygon"]

    clip = inside.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
    src = affected.geometry.to_numpy()
    has = clip.notna().to_numpy()
    rest = src.copy()
    rest[has] = shapely.difference(src[has], clip[has].to_numpy())
    rest = affected.assign(geometry=rest).explode(index_parts=False)
    rest = rest[(rest.geom_type == "Polygon") & ~rest.geometry.is_empty]

    inside["nin_kode_fkid"] = code
    inside["NiN_name"] = mnk.rules.NIN_NAMES[code]
    inside["nkm_prosess"] = mnk.attributes.NKM_PROSESS[code]
    inside["metode_klassifisering"] = method

    cols = [c for c in unaffected.columns if not c.startswith("_")]
    out = gpd.GeoDataFrame(
        pd.concat([unaffected[cols], rest[cols], inside[cols]], ignore_index=True), crs=CRS
    )
    print(f"{code}: affected {len(affected):,} -> {len(inside):,} pieces, "
          f"{inside.area.sum() / 1e6:,.1f} km²")
    return out

## 3. Apply direct rules in order

Each rule in `mnk.rules.DIRECT_RULES` is burned in; a later rule overrides an
earlier one where they overlap.

In [4]:
for rule in mnk.rules.DIRECT_RULES:
    mask = load_mask(rule)
    candidates = None
    if rule["where"]:
        candidates = np.logical_and.reduce(
            [final[col].isin(vals) for col, vals in rule["where"].items()]
        )
        candidates = pd.Series(candidates, index=final.index)
    final = override(
        final, mask, rule["nin_kode_fkid"],
        method=rule["metode_klassifisering"],
        candidates=candidates,
    )
    del mask, candidates
    gc.collect()

print(f"Combined: {len(final):,} polygons")
print(final["nin_kode_fkid"].value_counts())

NA-MC08: 106 polygons (181.5 km²)


NA-MC08: affected 3,046 -> 4,708 pieces, 152.8 km²


NA-MJ01: 3,846 polygons (207.8 km²)


NA-MJ01: affected 7,076 -> 14,660 pieces, 208.5 km²
Combined: 5,119,525 polygons
nin_kode_fkid
NA-MA01    1683947
NA-MA04     552835
NA-MA02     407811
NA-MA05     157864
NA-MA03     154238
NA-MA06      73502
NA-MB02      18545
NA-MC01      17851
NA-MJ01      14660
NA-MC04      13428
NA-MC02       5248
NA-MC08       4708
NA-MF01         14
Name: count, dtype: int64


## 4. Save ungridded output

The combined result is saved before the grid split as
`marint-naturkart-ungridded_norge_2026_25833.geo.parquet`. It has far fewer polygons
and no grid seams, so it is the source for the PMTiles (`scripts/pyramids.sh`).

In [5]:
ungridded_fname = mnk.utils.to_filename("marint-naturkart-ungridded", "norge", "2026", CRS.split(":")[1])
ungridded_path = REPO / f"{ungridded_fname}.gpkg"
ungridded_path.unlink(missing_ok=True)

for start in range(0, len(final), CHUNK_SAVE := 200_000):
    final.iloc[start : start + CHUNK_SAVE].to_file(
        ungridded_path, driver="GPKG", layer="nin", mode="w" if start == 0 else "a"
    )
    print(f"  {min(start + CHUNK_SAVE, len(final)):,}/{len(final):,}")
print(f"Saved {ungridded_path.name}: {len(final):,} polygons")

  200,000/5,119,525


  400,000/5,119,525


  600,000/5,119,525


  800,000/5,119,525


  1,000,000/5,119,525


  1,200,000/5,119,525


... (30 outputs left out)

/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


  5,000,000/5,119,525


  5,119,525/5,119,525
Saved marint-naturkart-ungridded_norge_2026_25833.gpkg: 5,119,525 polygons


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


## 5. Split by SSB 1 km statistical grid and save

Polygons are split along the SSB 1 km grid cells and tagged with `rute_fkid`
(`ssbid1000M`). `nk_fid` is renewed so every split polygon gets a unique ID.

**Memory**: the split multiplies the number of polygons, so it is done in chunks
of `CHUNK` input polygons. Each chunk is split and written straight to disk
(GPKG append + one parquet part per chunk); the parts are then streamed into a
single GeoParquet file. Polygons lying inside a single grid cell keep their
geometry without an intersection.

In [6]:
CHUNK = 50_000

grid = (
    gpd.read_parquet(GRID_URL, columns=["ssbid1000M", "geometry"])
    .rename(columns={"ssbid1000M": "rute_fkid"})
    .to_crs(CRS)
)
minx, miny, maxx, maxy = final.total_bounds
grid = grid.cx[minx:maxx, miny:maxy].reset_index(drop=True)
grid_geom = grid.geometry.to_numpy()
grid_id = grid["rute_fkid"].to_numpy()
grid_sindex = grid.sindex
print(f"Grid cells in extent: {len(grid):,}")

cols = [c for c in final.columns if c != "geometry"]
if "rute_fkid" not in cols:
    cols.insert(cols.index("opphav_kode") + 1, "rute_fkid")


def split_chunk(chunk):
    """Split *chunk* by the grid; returns exploded polygons with rute_fkid."""
    src_idx, cell_idx = grid_sindex.query(chunk.geometry, predicate="intersects")
    geoms = chunk.geometry.to_numpy()[src_idx]
    cells = grid_geom[cell_idx]
    # Skip the expensive intersection where the polygon is inside one cell.
    inside = shapely.contains_properly(cells, geoms) | shapely.covers(cells, geoms)
    out_geom = geoms.copy()
    out_geom[~inside] = shapely.intersection(geoms[~inside], cells[~inside])
    out = chunk.iloc[src_idx].drop(columns="geometry").reset_index(drop=True)
    out["rute_fkid"] = grid_id[cell_idx]
    out = gpd.GeoDataFrame(out, geometry=out_geom, crs=CRS)
    out = out.explode(index_parts=False)
    out = out[(out.geom_type == "Polygon") & ~out.geometry.is_empty]
    return out.reset_index(drop=True)

Grid cells in extent: 471,344


## 6. Save

In [7]:
fname = mnk.utils.to_filename("marint-naturkart", "norge", "2026", CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"
gpkg_path = REPO / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)

n_out = 0
with tempfile.TemporaryDirectory(dir=REPO) as tmp:
    parts = []
    for start in range(0, len(final), CHUNK):
        out = split_chunk(final.iloc[start : start + CHUNK])
        out["nk_fid"] = [f"NKM-{i:010d}" for i in range(n_out + 1, n_out + len(out) + 1)]
        out = out[cols + ["geometry"]]
        n_out += len(out)

        part = Path(tmp) / f"part_{start:010d}.parquet"
        out.to_parquet(part, compression="snappy")
        parts.append(part)
        out.to_file(gpkg_path, driver="GPKG", layer="nin", mode="w" if start == 0 else "a")
        print(f"  {min(start + CHUNK, len(final)):,}/{len(final):,} -> {n_out:,} polygons")
        del out
        gc.collect()

    # Stream parts into one GeoParquet file (unified schema, geo metadata kept).
    schemas = [pq.read_schema(p) for p in parts]
    schema = pa.unify_schemas(schemas, promote_options="permissive")
    geo = json.loads(schemas[0].metadata[b"geo"])
    for col in geo["columns"].values():
        col.pop("bbox", None)
    schema = schema.with_metadata({b"geo": json.dumps(geo).encode()})
    with pq.ParquetWriter(parquet_path, schema, compression="snappy") as writer:
        for p in parts:
            pf = pq.ParquetFile(p)
            for rg in range(pf.num_row_groups):
                writer.write_table(pf.read_row_group(rg).cast(schema))

print(f"Saved {n_out:,} polygons -> {parquet_path.name}, {gpkg_path.name}")
del final, grid, grid_geom, grid_id, grid_sindex
gc.collect()

  50,000/5,119,525 -> 107,789 polygons


  100,000/5,119,525 -> 196,148 polygons


  150,000/5,119,525 -> 301,377 polygons


  200,000/5,119,525 -> 409,428 polygons


  250,000/5,119,525 -> 521,289 polygons


  300,000/5,119,525 -> 639,306 polygons


... (95 outputs left out)

  5,100,000/5,119,525 -> 7,009,229 polygons


  5,119,525/5,119,525 -> 7,032,513 polygons


Saved 7,032,513 polygons -> marint-naturkart_norge_2026_25833.geo.parquet, marint-naturkart_norge_2026_25833.gpkg


206